# 深度循環神經網路
:label:`sec_deep_rnn`

到目前為止，我們只討論了具有一個單向隱藏層的循環神經網路。
其中，隱變量和觀測值與具體的函數形式的交互方式是相當隨意的。
只要交互類型建模具有足夠的靈活性，這就不是一個大問題。
然而，對一個單層來說，這可能具有相當的挑戰性。
之前在線性模型中，我們通過添加更多的層來解決這個問題。
而在循環神經網路中，我們首先需要確定如何添加更多的層，
以及在哪裡添加額外的非線性，因此這個問題有點棘手。

事實上，我們可以將多層循環神經網路堆疊在一起，
通過對幾個簡單層的組合，產生了一個靈活的機制。
特別是，數據可能與不同層的堆疊有關。
例如，我們可能希望保持有關金融市場狀況
（熊市或牛市）的宏觀數據可用，
而微觀數據只記錄較短期的時間動態。

![深度循環神經網路結構](../img/deep-rnn.svg)

## 函數依賴關係

我們可以將深度架構中的函數依賴關係形式化。
假設在時間步$t$有一個小批量的輸入數據
$\mathbf{X}_t \in \mathbb{R}^{n \times d}$
（樣本數：$n$，每個樣本中的輸入數：$d$）。
同時，將$l^\mathrm{th}$隱藏層（$l=1,\ldots,L$）
的隱狀態設為$\mathbf{H}_t^{(l)}  \in \mathbb{R}^{n \times h}$
（隱藏單元數：$h$），
輸出層變量設為$\mathbf{O}_t \in \mathbb{R}^{n \times q}$
（輸出數：$q$）。
設置$\mathbf{H}_t^{(0)} = \mathbf{X}_t$，
第$l$個隱藏層的隱狀態使用激活函數$\phi_l$，則：

$$\mathbf{H}_t^{(l)} = \phi_l(\mathbf{H}_t^{(l-1)} \mathbf{W}_{xh}^{(l)} + \mathbf{H}_{t-1}^{(l)} \mathbf{W}_{hh}^{(l)}  + \mathbf{b}_h^{(l)})$$

其中，權重$\mathbf{W}_{xh}^{(l)} \in \mathbb{R}^{h \times h}$，
$\mathbf{W}_{hh}^{(l)} \in \mathbb{R}^{h \times h}$和
偏置$\mathbf{b}_h^{(l)} \in \mathbb{R}^{1 \times h}$
都是第$l$個隱藏層的模型參數。

最後，輸出層的計算僅基於第$l$個隱藏層最終的隱狀態：

$$\mathbf{O}_t = \mathbf{H}_t^{(L)} \mathbf{W}_{hq} + \mathbf{b}_q$$

In [ ]:
import torch
from torch import nn
from d2l import torch as d2l

batch_size, num_steps = 32, 35
train_iter, vocab = d2l.load_data_time_machine(batch_size, num_steps)

## 簡潔實現

### [**使用多層LSTM模型**]

實現多層循環神經網路所需的許多邏輯細節在高級API中都是現成的。
我們使用一個含有兩個隱藏層的長短期記憶網路來訓練語言模型。

In [ ]:
vocab_size, num_hiddens, num_layers = len(vocab), 256, 2
num_inputs = vocab_size
device = d2l.try_gpu()
lstm_layer = nn.LSTM(num_inputs, num_hiddens, num_layers)
model = d2l.RNNModel(lstm_layer, len(vocab))
model = model.to(device)

### [**訓練與預測**]

In [ ]:
num_epochs, lr = 500, 2
d2l.train_ch8(model, train_iter, vocab, lr*1.0, num_epochs, device)

### 比較不同層數的效果

讓我們比較不同層數（1層、2層、3層）對模型性能的影響。

In [ ]:
# 比較不同深度的模型
def compare_depths(depths=[1, 2, 3], num_hiddens=256, num_epochs=100):
    """比較不同深度的RNN模型"""
    results = []
    
    for depth in depths:
        print(f"\n訓練 {depth} 層LSTM...")
        lstm_layer = nn.LSTM(len(vocab), num_hiddens, depth)
        model = d2l.RNNModel(lstm_layer, len(vocab))
        model = model.to(device)
        
        # 訓練模型
        d2l.train_ch8(model, train_iter, vocab, 2, num_epochs, device)
        
    print("\n深度RNN模型訓練完成！")

# 運行比較（可選）
# compare_depths()

## 🤖 AI 輔助學習指南

### 💡 核心概念理解

**Q: 為什麼要使用多層RNN而不是單層？**

A: 多層RNN的優勢：
1. **更強的表達能力**：多層可以學習到更複雜的特徵層次結構
2. **抽象層次**：底層捕捉低級特徵（如字符），高層捕捉高級特徵（如語義）
3. **非線性變換**：每一層都增加了額外的非線性，增強模型的擬合能力

**Q: 多層RNN會帶來什麼問題？**

A: 主要挑戰：
1. **訓練難度增加**：梯度消失/爆炸問題更嚴重
2. **計算成本高**：參數量和計算量都大幅增加
3. **容易過擬合**：模型容量大，需要更多數據和正則化

### 🔧 調試技巧

1. **梯度檢查**：使用 `torch.nn.utils.clip_grad_norm_` 防止梯度爆炸
2. **監控訓練**：觀察每層的激活值和梯度，確保信息流動正常
3. **從淺到深**：先訓練淺層網絡，再逐步增加層數

### 📊 性能優化建議

1. **層數選擇**：通常2-3層就足夠，超過4層收益遞減
2. **隱藏單元數**：每層可以使用不同的隱藏單元數
3. **Dropout**：在層與層之間添加dropout防止過擬合
4. **殘差連接**：對於很深的網絡，考慮使用殘差連接

### 🎯 實戰建議

**何時使用深度RNN：**
- 數據量充足（至少幾十萬樣本）
- 任務複雜度高（如機器翻譯、複雜序列建模）
- 有足夠的計算資源

**何時使用淺層RNN：**
- 數據量有限
- 任務相對簡單
- 需要快速訓練和推理

### 📈 常見錯誤與解決方案

| 問題 | 原因 | 解決方案 |
|------|------|----------|
| 損失不下降 | 學習率過大/過小 | 嘗試不同學習率，使用學習率調度器 |
| 訓練過慢 | 模型過深 | 減少層數或使用更高效的模型（如Transformer） |
| 過擬合 | 模型容量過大 | 添加dropout、減少層數或隱藏單元數 |
| 梯度爆炸 | 深度過大 | 使用梯度裁剪、減少層數 |

## 小結

* 在深度循環神經網路中，隱狀態的信息被傳遞到當前層的下一時間步和下一層的當前時間步。
* 有許多不同風格的深度循環神經網路，如長短期記憶網路、門控循環單元、或經典循環神經網路。
* 總體而言，深度循環神經網路需要大量的調參（如學習率和修剪）來確保合適的收斂，模型的初始化也需要謹慎。

## 練習

1. 基於我們在之前章節討論的單層實現，嘗試從零開始實現兩層循環神經網路。
2. 在本節訓練模型中，比較使用門控循環單元替換長短期記憶網路後模型的精確度和訓練速度。
3. 如果增加訓練數據，能夠將困惑度降到多低？
4. 在為文本建模時，是否可以將不同作者的源數據合併？有何優劣呢？

## 練習提示

**練習1提示：** 創建一個包含兩個隱藏層的RNN類，每層都維護自己的隱狀態。

**練習2提示：** 使用 `nn.GRU` 替換 `nn.LSTM`，保持其他參數不變進行對比。

**練習3提示：** 下載更大的文本語料庫，觀察困惑度的變化趨勢。

**練習4提示：** 考慮不同作者的寫作風格差異，可能需要添加作者標識作為額外輸入。